In [44]:
import pandas as pd
import ast
import networkx as nx
from shapely.geometry import Polygon
from utils.geometry_utils import rebuild_polygons
from utils.graph_utils import build_graph

In [45]:
rooms_df = pd.read_csv("rooms_dataset.csv")
rooms_df.head()

,room_type,area,centroid_x,centroid_y,num_vertices,points,plan_id
0,Outdoor Terrace,248496.0296,1260.503333,387.083333,6,"[(1030.81, 507.29), (1030.81, 146.67), (1719.8...",6044
1,Entry Lobby,54825.5028,1414.466667,1211.093333,9,"[(1469.34, 1137.48), (1469.34, 1148.48), (1530...",6044
2,Kitchen,67064.8976,1576.103333,1320.606667,6,"[(1712.49, 1478.05), (1474.05, 1478.05), (1474...",6044
3,LivingRoom,221476.7801,1482.916667,934.763333,6,"[(1712.49, 1137.48), (1469.34, 1137.48), (1334...",6044
4,Utility Laundry,20826.7515,1121.873333,1377.933333,6,"[(1190.03, 1478.05), (1051.17, 1478.05), (1051...",6044


In [46]:
rooms_df["points"] = rooms_df["points"].apply(ast.literal_eval)

In [48]:
rooms_df["polygon"]=rooms_df["points"].apply(lambda pts:Polygon(pts).buffer(0))

In [59]:
def share_wall(poly1, poly2, tolerance=15):

    if not poly1.is_valid:
        poly1 = poly1.buffer(0)

    if not poly2.is_valid:
        poly2 = poly2.buffer(0)

    poly1_expanded = poly1.buffer(tolerance)
    poly2_expanded = poly2.buffer(tolerance)

    return poly1_expanded.intersects(poly2_expanded)

In [60]:
import networkx as nx

def build_graph(plan_rooms):

    G = nx.Graph()

    # Add nodes with attributes
    for i, room1 in plan_rooms.iterrows():

        G.add_node(
            i,
            room_type=room1["room_type"],
            area=room1["area"],
            centroid=(room1["centroid_x"], room1["centroid_y"])
        )

    # Add edges (adjacency)
    for i, room1 in plan_rooms.iterrows():
        for j, room2 in plan_rooms.iterrows():

            if i >= j:
                continue

            if share_wall(room1["polygon"], room2["polygon"]):
                G.add_edge(i, j)

    return G

In [61]:
plan_id=rooms_df.iloc[0]["plan_id"]
plan_rooms=rooms_df[rooms_df["plan_id"]==plan_id]
G=build_graph(plan_rooms)

print("Rooms:", G.number_of_nodes())
print("Connections:", G.number_of_edges())

Rooms: 10
Connections: 16


In [62]:
for edge in G.edges():
    print(edge)

(1, 2)
(1, 3)
(1, 5)
(1, 7)
(1, 8)
(1, 9)
(2, 3)
(2, 8)
(3, 6)
(3, 7)
(4, 5)
(4, 9)
(5, 7)
(5, 9)
(6, 7)
(8, 9)


In [63]:
for u, v in G.edges():

    r1 = plan_rooms.loc[u]["room_type"]
    r2 = plan_rooms.loc[v]["room_type"]

    print(r1, "<->", r2)

Entry Lobby <-> Kitchen
Entry Lobby <-> LivingRoom
Entry Lobby <-> Bath
Entry Lobby <-> Undefined
Entry Lobby <-> DraughtLobby
Entry Lobby <-> Bath
Kitchen <-> LivingRoom
Kitchen <-> DraughtLobby
LivingRoom <-> Bedroom
LivingRoom <-> Undefined
Utility Laundry <-> Bath
Utility Laundry <-> Bath
Bath <-> Undefined
Bath <-> Bath
Bedroom <-> Undefined
DraughtLobby <-> Bath


In [64]:
graphs = {}

for plan_id in rooms_df["plan_id"].unique():

    plan_rooms = rooms_df[rooms_df["plan_id"] == plan_id]

    G = build_graph(plan_rooms)

    graphs[plan_id] = G

In [65]:
print("Total graphs:", len(graphs))

Total graphs: 4992
